# Step 7: Fine-tune ReactionT5 on a larger ORD pool (Model 1, Colab GPU)

Same `scripts/train_reactant_model_ord.py` as `colab/05_train_reactant_ord.ipynb`, but on a
**larger training pool** (~150,000 ORD reactions instead of 60,000, same seed/eval-exclusion
logic -- 0 leakage verified against `data/v2_ord_eval_targets.json`) to test whether more data
pushes accuracy past the v2 result (ORD exact_match 50.7%/top-5 74.3%, core_exact_match
62.3%/top-5 79.7%).

**Colab session budget varies (often ~2-3h/day per account).** Checkpoints are written to
Google Drive, and this notebook can simply be re-run later -- it auto-resumes from the last
checkpoint. Do not clear the Drive folder between sessions.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**Reclaim Drive quota (optional, run once per session):** the training script no
longer rotates/deletes checkpoints directly on Drive at all (see the v5 note below --
Trainer now checkpoints on local Colab disk and only ever *overwrites* one fixed
Drive folder), so this matters much less than before. Still useful once, to clear
out anything trashed by earlier (v2-v4) runs before this fix. First run prompts an
auth popup.

**Warning:** this empties Trash for your **entire** Google Drive account, not just
this project's files -- anything else you'd trashed elsewhere and might still want
to recover will be gone permanently too. Skip this cell if that matters to you.

In [ ]:
from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
build("drive", "v3").files().emptyTrash().execute()
print("Drive Trash emptied.")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

`data/v2_ord_train_150k/` is gitignored (large, derived) -- regenerate it deterministically
here (fixed seed, excludes the committed `data/v2_ord_eval_targets.json` by construction, so
this can never leak into the eval set). Building the 150k pool takes longer than the 60k one
used in `05_train_reactant_ord.ipynb` -- budget a few extra minutes for this cell, especially
on a cold Colab session with no cached ORD download yet. Only needs to run once per session;
skipped automatically if the files already exist (e.g. this is a same-day resume).

In [ ]:
import os

if not os.path.exists("data/v2_ord_train_150k/reactants_train.jsonl"):
    !python scripts/build_train_data_ord.py --pool-count 150000 --seed 42 --output-dir data/v2_ord_train_150k

**Why this notebook exists:** the 60k-pool run (`05_train_reactant_ord.ipynb`) went through
5 states (baseline -> v1 mix -> v2 ORD-only -> v3/v4/v5 SMILES augmentation) and settled on v2
as final -- see `RESULTS.md` at the repo root for the full numbers.

**Two corrections after the first two 150k attempts (see `RESULTS.md` sections 5-6):**
1. The first run launched without `--no-augment`, silently inheriting the script's default
   SMILES augmentation (prob=0.5) -- reproduced variant 4's regression on the bigger pool.
2. The *corrected* (no-augment) rerun still only matched v2 (48.7% vs 50.7% ORD exact_match
   top-1), not beaten it -- because it used the script's *current* defaults (`lr=2e-5`,
   `2 epochs`), not v2's actual proven config. Confirmed from v2's own saved `training_args.bin`:
   v2 used `lr=5e-5` and `3` epochs. Those defaults were lowered later, for v3/v4's
   augmentation-overfitting debugging, and never restored for the plain case -- so more data was
   never cleanly tested against the regimen that actually worked.

The cell below now passes `--no-augment --learning-rate 5e-5 --num-train-epochs 3` explicitly.
3 epochs over 147k examples means ~50% more steps than the earlier 150k attempts (~18,376 ->
~27,564 on a single GPU) -- expect more Colab sessions to reach completion.

**Cross-account resume:** Colab sessions may run under different Google accounts/Drives
each time (e.g. to spread GPU usage across accounts), so `{output_dir}/latest_checkpoint`
from a *previous* session isn't reliably visible to a *new* session automatically -- it's a
different Drive. Two ways to hand a checkpoint from one session to the next:

- **Recommended (faster for large checkpoints):** on drive.google.com (in whichever
  account is mounted *this* session), drag-and-drop the checkpoint folder (e.g. a
  downloaded `checkpoint-4750` or `final`) anywhere under My Drive -- the Drive website
  handles large-folder uploads far more reliably than a browser file picker. Then just
  type its path (under `/content/drive/MyDrive/...`) into `resume_from_checkpoint_path`
  in the next cell -- skip the upload-widget cell entirely.
- **Alternative:** the upload-widget cell below, if you'd rather not touch Drive directly
  (goes through the browser, slower for large folders -- fine for small ones).

Skip both if there's nothing to resume from yet (first run).

In [ ]:
resume_from_checkpoint_path = ""  # @param {type:"string"}
# e.g. /content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_ord150k/checkpoint-4750
# Leave blank if you're using the upload-widget cell below instead, or if this is a first run.

In [ ]:
import os
import shutil
import zipfile

from google.colab import files

uploaded = files.upload()  # skip this cell (don't run it) if you set resume_from_checkpoint_path above instead
if uploaded:
    zip_name = next(iter(uploaded))
    extract_dir = "/content/resume_from"
    shutil.rmtree(extract_dir, ignore_errors=True)
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(extract_dir)
    # A zip of a folder often nests one extra directory level (e.g. final/final/...);
    # if so, point at the inner one so config.json etc. are found directly.
    entries = os.listdir(extract_dir)
    if len(entries) == 1 and os.path.isdir(os.path.join(extract_dir, entries[0])):
        extract_dir = os.path.join(extract_dir, entries[0])
    resume_from_checkpoint_path = extract_dir
    print(f"Will resume from: {resume_from_checkpoint_path}")
    print("Contents:", os.listdir(resume_from_checkpoint_path))

In [ ]:
output_dir = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_ord150k_v2cfg"  # @param {type:"string"}
time_budget_minutes = 105  # @param {type:"number"}
# Set this a bit under your actual session length (e.g. 105 for a ~2h session) to leave
# headroom for Drive mount / pip install / data build before the script's own save-and-stop
# kicks in cleanly instead of Colab killing the process mid-step. output_dir renamed (_v2cfg)
# so it can't be confused with the earlier, weaker-hyperparameter 150k run's Drive folder.

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
resume_flag = ["--resume-from-checkpoint", resume_from_checkpoint_path] if resume_from_checkpoint_path else []

!python scripts/train_reactant_model_ord.py \
    --train-file data/v2_ord_train_150k/reactants_train.jsonl \
    --val-file data/v2_ord_train_150k/reactants_val.jsonl \
    --no-augment \
    --learning-rate 5e-5 \
    --num-train-epochs 3 \
    --output-dir "{output_dir}" \
    --time-budget-minutes {time_budget_minutes} \
    {' '.join(resume_flag)} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Training output is redirected to `train.log` in `output_dir` (on Drive) instead of
printing in this cell -- a long run's per-step tqdm bar and log lines used to grow the
notebook's output DOM large enough to make the browser tab unresponsive after 1-2 hours,
even though the actual training was proceeding fine underneath. The cell above still
blocks until training stops (so Colab doesn't treat the runtime as idle), but prints
nothing while it runs. To check progress without waiting: open `train.log` directly in
Google Drive's own web preview (refresh it there) -- that works independently of the
Colab kernel, which stays busy running the cell above.

**To continue in a later session:** once this session ends (or you stop it), go to
`output_dir` on *this* session's Drive and download either `final` (fresh
optimizer/step-count next time) or the newest `checkpoint-N` under `latest_checkpoint`
(exact resume, optimizer state preserved) as a zip. Next session: run this notebook
from the top, upload that zip in the "Cross-account resume" cell above, then run this
cell -- it will continue from those weights regardless of which Google account/Drive is
mounted this time. (If you happen to reconnect under the *same* account/Drive as last
time, you can skip the upload entirely -- it'll auto-resume from
`{output_dir}/latest_checkpoint` on its own.) Once `trainer.train()` finishes (not just
time-budget-stopped), the final model is saved to `{output_dir}/final`.

**Evaluate when done** (locally, after downloading `final`):

```
python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model <downloaded_final_dir> \
    --num-beams 10 --output experiments/v2_model1_topk/ord150k_topk.json
```